In [1]:
import sys

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import f1_score, precision_score, recall_score
from tqdm import tqdm

sys.path.append('/pfs/work9/workspace/scratch/ma_faroesch-master-thesis/playground-uc3/master-thesis')
%cd '/pfs/work9/workspace/scratch/ma_faroesch-master-thesis/playground-uc3/master-thesis/'

/pfs/work9/workspace/scratch/ma_faroesch-master-thesis/playground-uc3/master-thesis


In [2]:
from architecture.extended_cbm import CBMWrapper, ExtendedCBMOutput, init_from_checkpoint
from cbm_datasets import Batch, get_dataloader, get_datasets
from utils.others import calculate_metrics

In [ ]:
def create_summary_table(df: pd.DataFrame, filter_tag:list[str], reference_measure:str) -> pd.DataFrame:
    """Erstellt eine Tabelle mit den Bestwerten pro Run."""
    print("\nErstelle Summary Tabelle (Max-Werte)...")

    # filter df for each column where df[column] is true
    for tag in filter_tag:
        col = f"tag_{tag}"
        if col not in df.columns:
            raise ValueError(f"Tag-Spalte {col} existiert nicht im DataFrame.")
        df = df[df[col] == True]

    if df.empty:
        print("DataFrame nach Tag-Filter leer.")
        return df

    # Sicherstellen, dass innerhalb jedes Runs nach step sortiert ist
    df = df.sort_values(["run_id", "_step"]).copy()

    # Epoch als laufende Evaluation pro run_id (startet bei 0)
    df["epoch"] = df.groupby("run_id").cumcount()

    # Bestes F1-Concept pro Run auswählen
    best_per_run = (
        df.loc[df.groupby("run_id")[reference_measure].idxmax()]
        .reset_index(drop=True)
    )
    
    return best_per_run

In [94]:
data = pd.read_parquet(DATA_PATH)
data[data['tag_aab-con-maskreg-intro']==True]['run_id'].nunique()

3

In [90]:
DATA_PATH = 'thesis-figures/extended_cbm/results/1_extended_cbm_results.parquet'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
data = pd.read_parquet(DATA_PATH)

print("Using device:", device)

Using device: cuda


In [5]:
data.columns

Index(['_step', 'Recall Concepts', 'Precision Concepts', 'Total Loss',
       'Mean IoU', 'Foreground Dice', '$F_1$-Score Concepts',
       '$F_1$-Score Labels', 'Mean Dice', 'Concept Accuracy', 'Label Accuracy',
       'Runtime', 'normalized_epoch', 'epoch', 'tag_Final',
       'tag_aab-seg-benchmark', 'run_id', 'Concept Module',
       'Segmentation Module', 'Concept Criterion', 'Use Soft Labels',
       'unified_model', 'Dataset', 'Concept Loss', 'tag_aab-con',
       'tag_aab-con-maskreg-intro', 'tag_aab-soft-bce', 'tag_aab-aff',
       'tag_joint', 'tag_con-implicit-es', 'tag_con-implicit-cs'],
      dtype='object')

In [56]:
from dataclasses import asdict, dataclass
from typing import Any


@dataclass
class EvalMetrics:
    # --- core eval metrics ---
    loss: float
    dice_mean: float
    iou_mean: float
    f1_concept_activations: float
    precision_concepts: float
    recall_concepts: float
    accuracy_concepts: float
    f1_labels: float
    accuracy_labels: float
    foreground_dice_scores: float

    # --- optional metadata (for dataframe concatenation later) ---
    concept_module: str | None = None
    segmentation_module: str | None = None
    dataset: str | None = None
    runtime: float | None = None

    # -------------------------
    # 2️⃣ Pretty renamed dict
    # -------------------------
    def to_pretty_dict(self) -> dict[str, Any]:
        return {
            "Total Loss": self.loss,
            "Mean Dice": self.dice_mean,
            "Mean IoU": self.iou_mean,
            r"Concept Activations $F_1$-Score": self.f1_concept_activations,
            "Precision Concepts": self.precision_concepts,
            "Recall Concepts": self.recall_concepts,
            "Concept Accuracy": self.accuracy_concepts,
            r"Label $F_1$-Score": self.f1_labels,
            "Label Accuracy": self.accuracy_labels,
            "Foreground Dice": self.foreground_dice_scores,
            "Concept Module": self.concept_module,
            "Segmentation Module": self.segmentation_module,
            "Dataset": self.dataset,
            "Runtime": self.runtime,
        }

    # -------------------------
    # 3️⃣ DataFrame row
    # -------------------------
    def to_dataframe(self) -> pd.DataFrame:
        return pd.DataFrame([self.to_pretty_dict()])

In [57]:
@torch.no_grad()
def evaluate(
    model: CBMWrapper,
    val_loader: torch.utils.data.DataLoader,
    device: str,
    calculate_dice: bool,
    calculate_fg_dice: bool,
    calculate_iou: bool,
    concept_names: list[str],
    part_names: list[str],
    dice_thr: float = 0.5,
):
    model.eval()
    total_loss = 0.0
    total_samples = 0  # Trackt die tatsächliche Anzahl der Bilder

    dice_scores: list[float] = []
    iou_scores: list[float] = []
    foreground_dice_scores: list[float] = []
    f1_labels: list[float] = []

    all_concept_logits = []
    all_concept_targets = []

    all_label_preds = []
    all_label_targets = []


    correct = 0

    pbar = tqdm(val_loader, desc="[Eval]")
    for step, batch in enumerate(pbar):
        
        batch: Batch
        batch = batch.to(device)
        
        # Batch Größe ermitteln (wichtig für batch_size > 1)
        current_batch_size = batch.images.size(0)
        total_samples += current_batch_size

        predictions: ExtendedCBMOutput = model(batch.images)
        
        metrics = calculate_metrics(
            predictions,
            batch,
            mode="val",
            dice_thr=dice_thr,
            concept_names=concept_names,
            part_names=part_names,
            calculate_dice=calculate_dice,
            calculate_fg_dice=calculate_fg_dice,
            calculate_iou=calculate_iou,
            calculate_f1_concepts=False
        )

        # Accuracy Berechnung korrigiert für Batches
        pred_labels = predictions.classification_module.labels_logits.argmax(dim=1)
        gt_labels = batch.labels.argmax(dim=1)
        correct += (pred_labels == gt_labels).sum().item()

        # Metriken (calculate_metrics gibt meist schon Durchschnitte pro Batch zurück)
        if calculate_dice:
            dice_scores.append(metrics["val/dice_mean"])
        if calculate_fg_dice:
            foreground_dice_scores.append(metrics["val/foreground_dice_mean"])

        if calculate_iou:
            iou_scores.append(metrics["val/iou_mean"])
        
        all_concept_logits.append(
            predictions.concept_module.concept_logits.detach().cpu()
        )
        if batch.concepts is not None:
            all_concept_targets.append(
                batch.concepts.detach().cpu()
            )

        all_label_preds.append(pred_labels.cpu())
        all_label_targets.append(gt_labels.cpu())


    # Finale Berechnungen basierend auf total_samples statt len(val_loader)
    avg_loss = total_loss / total_samples

    avg_accuracy_labels = correct / total_samples

    # Da metrics meist Batch-Averages sind, ist np.mean über die Liste okay,
    # solange die Batches etwa gleich groß sind.
    avg_dice = float(np.mean(dice_scores)) if dice_scores else 0.0
    avg_iou = float(np.mean(iou_scores)) if iou_scores else 0.0
    avg_foreground_dice = float(np.mean(foreground_dice_scores)) if foreground_dice_scores else 0.0

    if all_concept_logits and False:
        concept_logits = torch.cat(all_concept_logits, dim=0)
        concept_targets = torch.cat(all_concept_targets, dim=0)

        # Wahrscheinlichkeiten und Vorhersagen (Threshold 0.5)
        probs = torch.sigmoid(concept_logits)
        preds = (probs >= 0.5).int().cpu().numpy()
        targets = (concept_targets >= 0.5).int().cpu().numpy()

        # Wir berechnen 'macro', um jedes Konzept gleich zu gewichten
        prec_concepts = float(precision_score(targets, preds, average="macro", zero_division=0))
        rec_concepts = float(recall_score(targets, preds, average="macro", zero_division=0))
        f1_concepts = float(f1_score(targets, preds, average="macro", zero_division=0))

        acc_per_concept = (preds == targets).mean(axis=0)
        acc_concepts = float(acc_per_concept.mean())
    else:
        prec_concepts = rec_concepts = f1_concepts = acc_concepts = 0.0

    if all_label_preds:

        label_preds = torch.cat(all_label_preds, dim=0)  # [N,]
        label_targets = torch.cat(all_label_targets, dim=0)  # [N,]
        f1_labels = f1_score(label_targets, label_preds, average="micro") # type: ignore
    else:
        f1_labels = 0

    print(f"Correct: {correct} of {total_samples} (Acc: {avg_accuracy_labels:.4f})")

    
    return EvalMetrics(
        loss=avg_loss,
        dice_mean=avg_dice,
        iou_mean=avg_iou,
        f1_concept_activations=f1_concepts,
        precision_concepts=prec_concepts,
        recall_concepts=rec_concepts,
        accuracy_concepts=acc_concepts,
        f1_labels=f1_labels,
        accuracy_labels=avg_accuracy_labels,
        foreground_dice_scores=avg_foreground_dice,
    )
    
    

In [58]:
import os


def calc_test_metrics(data, tags:list[str], n_runs:int, reference_measure:str, test_results_path:str): 
    
    if os.path.exists(test_results_path):
        df_test_results = pd.read_parquet(test_results_path)
        print('File exists')
    else:
        df_test_results = pd.DataFrame(columns=['run_id'])

    segmentation_mapping = {
        "segdino_b": "SegDINO",
        "SegmentationHeadSETRPUP": "SETR-PUP",
        "SegmentationHeadUpscaledSingle": "Upscaled SingleLayer",
        "Upscaled MultiLayer": "Upscaled MultiLayer",
    }

    concept_mapping = {
        "AvgPool": "Global Average Pooling",
        "AvgPool+Affine": "Affine-Calibrated Global Average Pooling",
        "Top-K AvgPool": r"Top-$\rho$ Average Pooling",
        "Maximum Pooling": "Maximum Pooling",
        "SoftmaxWeightedPooling": "Spatial Softmax Attention Pooling",
    }

    
    summary = create_summary_table(data, filter_tag=tags, reference_measure=reference_measure)

    assert len(summary) == n_runs, f"Erwartet {n_runs} Runs in der Summary Tabelle, aber gefunden: {len(summary)}"

    for _, row in summary.iterrows():
        dataset = row['Dataset'].lower()
        run_id = row['run_id']
        epoch = row['epoch']

        checkpoint_path = f"/pfs/work9/workspace/scratch/ma_faroesch-master-thesis/blobs/checkpoints/cbm_anyup_dinov3_{dataset}_run_{run_id}_epoch{epoch}.pth"
    

        if not os.path.exists(checkpoint_path):
            raise FileNotFoundError(f"Checkpoint {checkpoint_path} existiert nicht.")
        
    print(f"Berechne Test-Metriken für {len(summary)} Runs...")
        

        
    for _, row in summary.iterrows():
        if row['run_id'] in df_test_results['run_id'].values:
            
            # if Concept Module or Segmentation Module are None, overwrite them in df_test_results with values from summary
            # if pd.isna(df_test_results.loc[df_test_results['run_id'] == row['run_id'], 'Concept Module']).all():
            #     df_test_results.loc[df_test_results['run_id'] == row['run_id'], 'Concept Module'] = concept_mapping[row['Concept Module']]
            if pd.isna(df_test_results.loc[df_test_results['run_id'] == row['run_id'], 'Segmentation Module']).all():
                df_test_results.loc[df_test_results['run_id'] == row['run_id'], 'Segmentation Module'] = segmentation_mapping[row['Segmentation Module']]

            print(f"Skip {row['run_id']} (bereits in Test-Ergebnissen vorhanden)")
            continue
        print(f'Berechne {row["run_id"]}')

        
        train_dataset, val_dataset, test_dataset, n_concepts, n_classes, _ = get_datasets(dataset_name=row['Dataset'], 
                                                            root_dir='/pfs/work9/workspace/scratch/ma_faroesch-master-thesis/datasets', 
                                                            img_size=256, 
                                                            use_soft_labels=False, concept_masks_scale=None,
                                                            attr_level='image')

        _, _, test_loader = get_dataloader(batch_size=4, num_workers=14, 
                                            train_dataset=train_dataset, val_dataset=val_dataset, 
                                            test_dataset=test_dataset)
        
        model = init_from_checkpoint(run_id=row['run_id'], epoch=row['epoch'], dataset=row['Dataset'].lower(), device=device)
        model = model.to(device)
        
        results = evaluate(
            model=model,
            val_loader=test_loader,
            device=str(device),
            calculate_dice=False,
            calculate_fg_dice=False,
            calculate_iou=False,
            concept_names=[f"concept_{i}" for i in range(n_concepts)],
            part_names=[f"part_{i}" for i in range(n_classes)],
            dice_thr=0.5
        )
        results = results.to_dataframe()
        results['run_id'] = row['run_id']
        results['epoch'] = row['epoch']
        results['dataset'] = row['Dataset']
        df_test_results = pd.concat([df_test_results, results])



    

    print(df_test_results["Concept Module"].unique())

    df_test_results["Concept Module"] = pd.Categorical(
        df_test_results["Concept Module"].replace(concept_mapping),
        categories=list(concept_mapping.values()),
        ordered=True
    )

    # 2️⃣ Segmentation Module ebenfalls
    df_test_results["Segmentation Module"] = pd.Categorical(
        df_test_results["Segmentation Module"].replace(segmentation_mapping),
        categories=list(segmentation_mapping.values()),
        ordered=True
    )

    # 3️⃣ Sortieren
    df_sorted = df_test_results.sort_values(
        by=["Concept Module", "Segmentation Module"],
        ascending=True
    )


    return df_sorted

In [27]:
data.columns

Index(['_step', 'Recall Concepts', 'Precision Concepts', 'Total Loss',
       'Mean IoU', 'Foreground Dice', '$F_1$-Score Concepts',
       '$F_1$-Score Labels', 'Mean Dice', 'Concept Accuracy', 'Label Accuracy',
       'Runtime', 'normalized_epoch', 'epoch', 'tag_Final',
       'tag_aab-seg-benchmark', 'run_id', 'Concept Module',
       'Segmentation Module', 'Concept Criterion', 'Use Soft Labels',
       'unified_model', 'Dataset', 'Concept Loss', 'tag_aab-con',
       'tag_aab-con-maskreg-intro', 'tag_aab-soft-bce', 'tag_aab-aff',
       'tag_joint', 'tag_con-implicit-es', 'tag_con-implicit-cs'],
      dtype='object')

# AAB-CON

In [28]:
TEST_RESULTS_PATH_CON = 'thesis-figures/extended_cbm/results/2_test_metrics_aab_con.parquet'

df_test_results_aab_con = calc_test_metrics(data=data, tags=['aab-con'], n_runs=29, reference_measure=r"$F_1$-Score Concepts", test_results_path=TEST_RESULTS_PATH_CON)

File exists

Erstelle Summary Tabelle (Max-Werte)...
Berechne Test-Metriken für 29 Runs...
Skip 3vardl0g (bereits in Test-Ergebnissen vorhanden)
Skip 48x2ek26 (bereits in Test-Ergebnissen vorhanden)
Skip 652znd0o (bereits in Test-Ergebnissen vorhanden)
Skip 8jefa23q (bereits in Test-Ergebnissen vorhanden)
Skip 8s5i66xe (bereits in Test-Ergebnissen vorhanden)
Skip d8oxb1yg (bereits in Test-Ergebnissen vorhanden)
Skip dstfi0h2 (bereits in Test-Ergebnissen vorhanden)
Skip dwxi34vh (bereits in Test-Ergebnissen vorhanden)
Skip e2xe2ikz (bereits in Test-Ergebnissen vorhanden)
Skip ecb3hwum (bereits in Test-Ergebnissen vorhanden)
Skip frtwr5la (bereits in Test-Ergebnissen vorhanden)
Skip gv70wmzz (bereits in Test-Ergebnissen vorhanden)
Skip jguzfmll (bereits in Test-Ergebnissen vorhanden)
Skip k35hft9t (bereits in Test-Ergebnissen vorhanden)
Skip mn106ezy (bereits in Test-Ergebnissen vorhanden)
Skip n1ohshjk (bereits in Test-Ergebnissen vorhanden)
Skip n4lrcx2q (bereits in Test-Ergebnissen vo

/scratch/slurm_tmpdir/job_3442548/ipykernel_2524493/1943250257.py:97: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  df_test_results["Concept Module"].replace(concept_mapping),


In [29]:
l = df_test_results_aab_con.to_latex(index=False, float_format="%.3f", column_format="llccccccc")
print(l)

\begin{tabular}{llccccccc}
\toprule
run_id & epoch & Total Loss & Mean Dice & Mean IoU & Concept Activations $F_1$-Score & Precision Concepts & Recall Concepts & Concept Accuracy & Label $F_1$-Score & Label Accuracy & Foreground Dice & Dataset & Concept Module & Segmentation Module \\
\midrule
k35hft9t & 16 & 0.000 & 0.802 & 0.801 & 0.990 & 0.992 & 0.987 & 0.997 & 0.029 & 0.029 & 0.011 & FunnyBirds & Global Average Pooling & SegDINO \\
jguzfmll & 17 & 0.000 & 0.770 & 0.769 & 0.947 & 0.958 & 0.940 & 0.987 & 0.029 & 0.029 & 0.010 & FunnyBirds & Global Average Pooling & SETR-PUP \\
e2xe2ikz & 21 & 0.000 & 0.009 & 0.007 & 0.749 & 0.802 & 0.723 & 0.934 & 0.020 & 0.020 & 0.012 & FunnyBirds & Global Average Pooling & Upscaled SingleLayer \\
pde5gcnk & 22 & 0.000 & 0.720 & 0.719 & 0.959 & 0.954 & 0.965 & 0.989 & 0.000 & 0.000 & 0.010 & FunnyBirds & Global Average Pooling & Upscaled MultiLayer \\
u4w58p5i & 18 & 0.000 & 0.801 & 0.800 & 0.986 & 0.989 & 0.984 & 0.996 & 0.020 & 0.020 & 0.010 & Fun

# AAB SEG

In [30]:
TEST_RESULTS_PATH_SEG = 'thesis-figures/extended_cbm/results/2_test_metrics_aab_seg.parquet'

df_test_results_aab_seg = calc_test_metrics(data=data, tags=['aab-seg-benchmark'], n_runs=8, reference_measure=r"Foreground Dice", test_results_path=TEST_RESULTS_PATH_SEG)

File exists

Erstelle Summary Tabelle (Max-Werte)...
Berechne Test-Metriken für 8 Runs...
Skip 1jpajf8d (bereits in Test-Ergebnissen vorhanden)
Skip 5bhtbcxc (bereits in Test-Ergebnissen vorhanden)
Skip av3igr7l (bereits in Test-Ergebnissen vorhanden)
Skip c01q4i6o (bereits in Test-Ergebnissen vorhanden)
Skip i6n2g0yg (bereits in Test-Ergebnissen vorhanden)
Skip pbvhti5j (bereits in Test-Ergebnissen vorhanden)
Skip q0gevqso (bereits in Test-Ergebnissen vorhanden)
Skip xlp9yh1g (bereits in Test-Ergebnissen vorhanden)
['Global Average Pooling']
Categories (5, object): ['Global Average Pooling' < 'Affine-Calibrated Global Average Pooling' < 'Top-$\rho$ Average Pooling' < 'Maximum Pooling' < 'Spatial Softmax Attention Pooling']


In [31]:
latex = df_test_results_aab_seg.to_latex(index=False, float_format="%.3f", column_format="llccccccc")
print(latex)

\begin{tabular}{llccccccc}
\toprule
run_id & Total Loss & Mean Dice & Mean IoU & Concept Activations $F_1$-Score & Precision Concepts & Recall Concepts & Concept Accuracy & Label $F_1$-Score & Label Accuracy & Foreground Dice & Concept Module & Segmentation Module & Dataset & Runtime & epoch & dataset \\
\midrule
1jpajf8d & 0.000 & 0.899 & 0.874 & 0.000 & 0.000 & 0.000 & 0.814 & 0.023 & 0.023 & 0.642 & Global Average Pooling & SegDINO & NaN & NaN & 23.000 & FunnyBirds \\
pbvhti5j & 0.000 & 0.451 & 0.424 & 0.000 & 0.000 & 0.000 & 0.773 & 0.003 & 0.003 & 0.511 & Global Average Pooling & SegDINO & NaN & NaN & 7.000 & CUB_112 \\
q0gevqso & 0.000 & 0.503 & 0.476 & 0.000 & 0.000 & 0.000 & 0.773 & 0.005 & 0.005 & 0.561 & Global Average Pooling & SETR-PUP & NaN & NaN & 12.000 & CUB_112 \\
xlp9yh1g & 0.000 & 0.931 & 0.913 & 0.000 & 0.000 & 0.000 & 0.814 & 0.014 & 0.014 & 0.821 & Global Average Pooling & SETR-PUP & NaN & NaN & 23.000 & FunnyBirds \\
5bhtbcxc & 0.000 & 0.522 & 0.497 & 0.000 & 0.0

In [32]:
df_test_results_aab_seg.to_parquet(TEST_RESULTS_PATH_SEG)

# AAB Mask Reg

3

In [91]:
TEST_RESULTS_PATH_MASKREG = 'thesis-figures/extended_cbm/results/2_test_metrics_aab_maskreg-intro.parquet'

df_test_results_aab_maskreg = calc_test_metrics(data=data, tags=['aab-con-maskreg-intro'], n_runs=3, reference_measure=r"$F_1$-Score Concepts", test_results_path=TEST_RESULTS_PATH_MASKREG)



File exists

Erstelle Summary Tabelle (Max-Werte)...
Berechne Test-Metriken für 3 Runs...
Skip cmxceof3 (bereits in Test-Ergebnissen vorhanden)
Skip g3c6b6bf (bereits in Test-Ergebnissen vorhanden)
Skip xkfv5cvz (bereits in Test-Ergebnissen vorhanden)
[NaN]
Categories (5, object): ['Global Average Pooling' < 'Affine-Calibrated Global Average Pooling' < 'Top-$\rho$ Average Pooling' < 'Maximum Pooling' < 'Spatial Softmax Attention Pooling']


In [34]:
df_test_results_aab_maskreg.head()

,run_id,Total Loss,Mean Dice,Mean IoU,Concept Activations $F_1$-Score,Precision Concepts,Recall Concepts,Concept Accuracy,Label $F_1$-Score,Label Accuracy,Foreground Dice,Concept Module,Segmentation Module,Dataset,Runtime,epoch,dataset
0,cmxceof3,0.0,0.744500,0.735050,0.532584,0.596500,0.501046,0.826284,0.001208,0.001208,0.158443,NaN,Upscaled MultiLayer,None,None,23.0,CUB_112
0,g3c6b6bf,0.0,0.578592,0.563525,0.529115,0.596785,0.496773,0.825785,0.001899,0.001899,0.224552,NaN,Upscaled MultiLayer,None,None,23.0,CUB_112
0,xkfv5cvz,0.0,0.724397,0.711519,0.536706,0.595529,0.505982,0.827372,0.002071,0.002071,0.216143,NaN,Upscaled MultiLayer,None,None,19.0,CUB_112


In [35]:
df_test_results_aab_maskreg.to_parquet(TEST_RESULTS_PATH_MASKREG)

In [36]:
l = df_test_results_aab_maskreg.to_latex()
print(l)

\begin{tabular}{llrrrrrrrrrrllllrl}
\toprule
 & run_id & Total Loss & Mean Dice & Mean IoU & Concept Activations $F_1$-Score & Precision Concepts & Recall Concepts & Concept Accuracy & Label $F_1$-Score & Label Accuracy & Foreground Dice & Concept Module & Segmentation Module & Dataset & Runtime & epoch & dataset \\
\midrule
0 & cmxceof3 & 0.000000 & 0.744500 & 0.735050 & 0.532584 & 0.596500 & 0.501046 & 0.826284 & 0.001208 & 0.001208 & 0.158443 & NaN & Upscaled MultiLayer & NaN & NaN & 23.000000 & CUB_112 \\
0 & g3c6b6bf & 0.000000 & 0.578592 & 0.563525 & 0.529115 & 0.596785 & 0.496773 & 0.825785 & 0.001899 & 0.001899 & 0.224552 & NaN & Upscaled MultiLayer & NaN & NaN & 23.000000 & CUB_112 \\
0 & xkfv5cvz & 0.000000 & 0.724397 & 0.711519 & 0.536706 & 0.595529 & 0.505982 & 0.827372 & 0.002071 & 0.002071 & 0.216143 & NaN & Upscaled MultiLayer & NaN & NaN & 19.000000 & CUB_112 \\
\bottomrule
\end{tabular}



# Soft BCE

In [37]:
data.columns

Index(['_step', 'Recall Concepts', 'Precision Concepts', 'Total Loss',
       'Mean IoU', 'Foreground Dice', '$F_1$-Score Concepts',
       '$F_1$-Score Labels', 'Mean Dice', 'Concept Accuracy', 'Label Accuracy',
       'Runtime', 'normalized_epoch', 'epoch', 'tag_Final',
       'tag_aab-seg-benchmark', 'run_id', 'Concept Module',
       'Segmentation Module', 'Concept Criterion', 'Use Soft Labels',
       'unified_model', 'Dataset', 'Concept Loss', 'tag_aab-con',
       'tag_aab-con-maskreg-intro', 'tag_aab-soft-bce', 'tag_aab-aff',
       'tag_joint', 'tag_con-implicit-es', 'tag_con-implicit-cs'],
      dtype='object')

In [38]:
TEST_RESULTS_PATH_SOFTBCE = 'thesis-figures/extended_cbm/results/2_test_metrics_aab-soft-bce.parquet'

df_test_results_aab_softbce = calc_test_metrics(data=data, tags=['aab-soft-bce'], n_runs=4, reference_measure=r"$F_1$-Score Concepts", test_results_path=TEST_RESULTS_PATH_SOFTBCE)



File exists

Erstelle Summary Tabelle (Max-Werte)...
Berechne Test-Metriken für 4 Runs...
Skip 17m6p70g (bereits in Test-Ergebnissen vorhanden)
Skip qatqo9db (bereits in Test-Ergebnissen vorhanden)
Skip quos7n74 (bereits in Test-Ergebnissen vorhanden)
Skip sltpp8ev (bereits in Test-Ergebnissen vorhanden)
['Top-$\rho$ Average Pooling']
Categories (5, object): ['Global Average Pooling' < 'Affine-Calibrated Global Average Pooling' < 'Top-$\rho$ Average Pooling' < 'Maximum Pooling' < 'Spatial Softmax Attention Pooling']


In [39]:
df_test_results_aab_softbce.to_parquet(TEST_RESULTS_PATH_SOFTBCE)

# Aff

In [40]:
TEST_RESULTS_PATH_AFF = 'thesis-figures/extended_cbm/results/2_test_metrics_aab-aff.parquet'

df_test_results_aab_aff = calc_test_metrics(data=data, tags=['aab-aff'], n_runs=3, reference_measure=r"$F_1$-Score Concepts", test_results_path=TEST_RESULTS_PATH_AFF)



File exists

Erstelle Summary Tabelle (Max-Werte)...
Berechne Test-Metriken für 3 Runs...
Skip 9lexo76a (bereits in Test-Ergebnissen vorhanden)
Skip c0ldpb7v (bereits in Test-Ergebnissen vorhanden)
Skip msu693b8 (bereits in Test-Ergebnissen vorhanden)
[NaN]
Categories (5, object): ['Global Average Pooling' < 'Affine-Calibrated Global Average Pooling' < 'Top-$\rho$ Average Pooling' < 'Maximum Pooling' < 'Spatial Softmax Attention Pooling']


In [41]:
df_test_results_aab_aff.to_parquet(TEST_RESULTS_PATH_AFF)

# Implicit ES

In [42]:
data.columns

Index(['_step', 'Recall Concepts', 'Precision Concepts', 'Total Loss',
       'Mean IoU', 'Foreground Dice', '$F_1$-Score Concepts',
       '$F_1$-Score Labels', 'Mean Dice', 'Concept Accuracy', 'Label Accuracy',
       'Runtime', 'normalized_epoch', 'epoch', 'tag_Final',
       'tag_aab-seg-benchmark', 'run_id', 'Concept Module',
       'Segmentation Module', 'Concept Criterion', 'Use Soft Labels',
       'unified_model', 'Dataset', 'Concept Loss', 'tag_aab-con',
       'tag_aab-con-maskreg-intro', 'tag_aab-soft-bce', 'tag_aab-aff',
       'tag_joint', 'tag_con-implicit-es', 'tag_con-implicit-cs'],
      dtype='object')

In [59]:
TEST_RESULTS_PATH_IMPLICIT_ES = 'thesis-figures/extended_cbm/results/2_test_metrics_aab-implicit.parquet'

df_test_results_con_implicit_es = calc_test_metrics(data=data, tags=['con-implicit-es'], n_runs=5, reference_measure=r"$F_1$-Score Labels", test_results_path=TEST_RESULTS_PATH_IMPLICIT_ES)




Erstelle Summary Tabelle (Max-Werte)...
Berechne Test-Metriken für 5 Runs...
Berechne 9i6fpi2h


Using cache found in /pfs/work9/workspace/scratch/ma_faroesch-master-thesis/.cache/hub/wimmerth_anyup_main
[Eval]: 100%|██████████| 1449/1449 [07:18<00:00,  3.31it/s]


Correct: 4738 of 5794 (Acc: 0.8177)
Berechne ewzmye18


Using cache found in /pfs/work9/workspace/scratch/ma_faroesch-master-thesis/.cache/hub/wimmerth_anyup_main
[Eval]: 100%|██████████| 1449/1449 [07:06<00:00,  3.40it/s]


Correct: 4818 of 5794 (Acc: 0.8315)
Berechne ij9iskh2


...

Correct: 4746 of 5794 (Acc: 0.8191)
Berechne k9anbolk


Using cache found in /pfs/work9/workspace/scratch/ma_faroesch-master-thesis/.cache/hub/wimmerth_anyup_main
[Eval]: 100%|██████████| 1449/1449 [07:12<00:00,  3.35it/s]


Correct: 4778 of 5794 (Acc: 0.8246)
Berechne tidjt3zg


Using cache found in /pfs/work9/workspace/scratch/ma_faroesch-master-thesis/.cache/hub/wimmerth_anyup_main
[Eval]: 100%|██████████| 1449/1449 [06:20<00:00,  3.80it/s]


Correct: 4775 of 5794 (Acc: 0.8241)
[None]


In [60]:
df_test_results_con_implicit_es.to_parquet(TEST_RESULTS_PATH_IMPLICIT_ES)

# Implicit CS

In [87]:
TEST_RESULTS_PATH_IMPLICIT_CS = 'thesis-figures/extended_cbm/results/t_metrics_aab-implicit-cs.parquet'

df_test_results_aab_con_implicit_cs = calc_test_metrics(data=data, tags=['con-implicit-cs'], n_runs=12, reference_measure=r"$F_1$-Score Labels", test_results_path=TEST_RESULTS_PATH_IMPLICIT_CS)




Erstelle Summary Tabelle (Max-Werte)...
Berechne Test-Metriken für 12 Runs...
Berechne 1fna33r7


Using cache found in /pfs/work9/workspace/scratch/ma_faroesch-master-thesis/.cache/hub/wimmerth_anyup_main
[Eval]: 100%|██████████| 1449/1449 [07:12<00:00,  3.35it/s]


Correct: 4770 of 5794 (Acc: 0.8233)
Berechne 251d1b17


Using cache found in /pfs/work9/workspace/scratch/ma_faroesch-master-thesis/.cache/hub/wimmerth_anyup_main
[Eval]: 100%|██████████| 88/88 [00:12<00:00,  6.87it/s]


Correct: 323 of 350 (Acc: 0.9229)
Berechne 4doi6sje


...

Correct: 317 of 350 (Acc: 0.9057)
Berechne xx7xk7x8


Using cache found in /pfs/work9/workspace/scratch/ma_faroesch-master-thesis/.cache/hub/wimmerth_anyup_main
[Eval]: 100%|██████████| 1449/1449 [06:00<00:00,  4.02it/s]


Correct: 4133 of 5794 (Acc: 0.7133)
Berechne yg33klw8


Using cache found in /pfs/work9/workspace/scratch/ma_faroesch-master-thesis/.cache/hub/wimmerth_anyup_main
[Eval]: 100%|██████████| 1449/1449 [04:12<00:00,  5.73it/s]


Correct: 3794 of 5794 (Acc: 0.6548)
[None]


In [81]:
concept = pd.DataFrame(
    [
        {'run_id':'1fna33r7','n_concepts':50},
        {'run_id':'251d1b17','n_concepts':5},
        {'run_id':'4doi6sje','n_concepts':20},
        {'run_id':'cc756c5n','n_concepts':3},
        {'run_id':'dow7bigo','n_concepts':25},
        {'run_id':'f1y119wh','n_concepts':112},
        {'run_id':'juky1qof','n_concepts':26},
        {'run_id':'mbkrvmu2','n_concepts':80},
        {'run_id':'q8g94l02','n_concepts':40},
        {'run_id':'qd2z0cq3','n_concepts':10},
        {'run_id':'xx7xk7x8','n_concepts':10},
        {'run_id':'yg33klw8','n_concepts':5},
    ]
)

In [83]:
data = data.merge(concept, on='run_id'
)


In [85]:
a = create_summary_table(data, ['con-implicit-cs'], "$F_1$-Score Labels")


Erstelle Summary Tabelle (Max-Werte)...


In [86]:
for i, row in a.iterrows():
    print(i)
    dataset = row['Dataset'].lower()
    run_id = row['run_id']
    epoch = row['epoch']

    checkpoint_path = f"/pfs/work9/workspace/scratch/ma_faroesch-master-thesis/blobs/checkpoints/cbm_anyup_dinov3_{dataset}_run_{run_id}_epoch{epoch}.pth"

    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    checkpoint['config']['n_concepts'] = row['n_concepts']
    torch.save(checkpoint, checkpoint_path)

0
1
2
3
4
5
6
7
8
9
10
11


In [88]:
df_test_results_aab_con_implicit_cs.to_parquet(TEST_RESULTS_PATH_IMPLICIT_CS)